In [ ]:
# [auto] project-root setup
import os, sys
from pathlib import Path

# Walk upward to find the project root (folder containing .gitignore)
_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / '.gitignore').exists():
    _p = _p.parent
PROJECT_ROOT = _p

# Switch cwd to the project root so every relative path (Stage1_Exploration/, Refined_Results_v4/, ...) keeps working
os.chdir(PROJECT_ROOT)
# Let the notebooks do `from viz_config import VizConfig`
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from pysr import PySRRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# ==========================================
# 0. Experiment configuration
# ==========================================
# Training-set sizes to sweep for the data-efficiency study
# From very small (100) to the full pool (44400)
DATA_SIZES = [
    100, 200, 300, 400, 500, 
    750, 1000, 2000, 3000, 4000, 
    5000, 7500, 10000, 20000, 30000, 44400
]

FIXED_NOISE = 0.5  # Fixed noise level (50%)
N_REPEATS = 20     # Number of repetitions per size for statistical significance
SCALE = 1e7        # Concentration scaling factor
FEATURE_NAMES = ['V_in', 'C_in', 'Area', 'Distance']

OUTPUT_DIR = "Data_Efficiency_Results"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ==========================================
# 1. Data preparation
# ==========================================
print("Loading data...")
df = pd.read_csv('data/train_dataset_ready.csv')

# Apply scaling
df['C_in'] = df['C_in'] * SCALE
df['C_out'] = df['C_out'] * SCALE

# Build a fixed test set
# Critical: the test set must be identical across every experiment for fair comparison
unique_cases = df['Case'].unique()
train_cases_pool, test_cases_fixed = train_test_split(unique_cases, test_size=0.2, random_state=42)

# Prepare clean test data (ground truth)
test_df = df[df['Case'].isin(test_cases_fixed)].copy()
X_test = test_df[FEATURE_NAMES].values
y_test_clean = test_df['C_out'].values

# Build the training pool
# We sample from this pool at different sizes according to DATA_SIZES
train_df_pool = df[df['Case'].isin(train_cases_pool)].copy()
train_pool_indices = np.arange(len(train_df_pool))

print(f"Data ready. Each experiment will be repeated {N_REPEATS} times.")

# ==========================================
# 2. Experiment loop
# ==========================================
results_summary = []

for size in DATA_SIZES:
    print(f"\n{'='*40}")
    print(f"Evaluating training size: {size}")
    print(f"{'='*40}")
    
    current_size = min(size, len(train_pool_indices))
    r2_scores_hybrid = []
    
    for i in range(N_REPEATS):
        # Vary the random seed so each repetition draws a different subset
        seed = 42 + size + i 
        np.random.seed(seed)
        
        # A. Random sampling
        sampled_indices = np.random.choice(train_pool_indices, current_size, replace=False)
        X_train_sub = train_df_pool.iloc[sampled_indices][FEATURE_NAMES].values
        y_train_clean_sub = train_df_pool.iloc[sampled_indices]['C_out'].values
        
        # B. Inject noise
        # Simulates high-noise real-world data (50% noise)
        noise = np.random.normal(0, FIXED_NOISE * y_train_clean_sub)
        y_train_noisy = np.maximum(y_train_clean_sub + noise, 1e-6)
        
        # C. Train the MLP denoiser
        scaler_X = StandardScaler()
        X_train_scaled = scaler_X.fit_transform(X_train_sub)
        
        mlp = MLPRegressor(
            hidden_layer_sizes=(128, 64, 32),
            activation='relu',
            learning_rate_init=0.001,
            max_iter=500, 
            random_state=seed
        )
        try:
            mlp.fit(X_train_scaled, y_train_noisy)
        except Exception as e:
            print(f"    [Warning] MLP fit failed: {e}")
            continue

        # D. PySR Hybrid PySR training
        # Cap the PySR input size at 2000 for speed
        pysr_n = min(current_size, 2000)
        pysr_idx = np.random.choice(len(X_train_sub), pysr_n, replace=False)
        
        X_pysr_train = X_train_sub[pysr_idx]
        X_pysr_train_scaled = X_train_scaled[pysr_idx]
        
        # Core step: pre-clean the PySR training data with the MLP"pre-cleaning"
        y_denoised = mlp.predict(X_pysr_train_scaled)
        y_denoised = np.maximum(y_denoised, 1e-6)
        
        # PySR Fit the denoised data
        model_hybrid = PySRRegressor(
            niterations=40, # Few iterations - we just need a quick sanity check
            populations=15,
            binary_operators=["+", "*", "-", "/"],
            unary_operators=["inv(x)=1/x", "sqrt", "square"],
            extra_sympy_mappings={'inv': lambda x: 1/x},
            verbosity=0,
            temp_equation_file=True,
            delete_tempfiles=True 
        )
        
        r2 = -1.0 # Default failure score
        try:
            model_hybrid.fit(X_pysr_train, y_denoised, variable_names=FEATURE_NAMES)
            
            # --- Safety check ---
            y_pred = model_hybrid.predict(X_test)
            
            # Check for Inf / NaN (generated formula may have singularities)
            if not np.all(np.isfinite(y_pred)):
                print(f"    Repeat {i+1}/{N_REPEATS} | [Warning] PySR Singular formula (Inf/NaN) - skipping this run.")
                r2 = 0.0 
            else:
                # Evaluate R^2 on the clean test set
                r2 = r2_score(y_test_clean, y_pred)
                print(f"    Repeat {i+1}/{N_REPEATS} | Hybrid R2: {r2:.4f}")
                
        except Exception as e:
            print(f"    Repeat {i+1}/{N_REPEATS} | [Error] PySR Predict Error: {e}")
            r2 = 0.0

        r2_scores_hybrid.append(r2)

    # E. Statistics
    valid_scores = [s for s in r2_scores_hybrid if s > 0.0]
    if len(valid_scores) > 0:
        mean_r2 = np.mean(valid_scores)
        std_r2 = np.std(valid_scores)
    else:
        mean_r2 = 0.0
        std_r2 = 0.0
    
    print(f"  >>> Size {current_size} Result: mean R^2 = {mean_r2:.4f} (std: {std_r2:.4f})")
    
    # Save the aggregated stats for this training size
    results_summary.append({
        'Training_Size': current_size,
        'Mean_R2': mean_r2,
        'Std_R2': std_r2,
        'Raw_Scores': str(r2_scores_hybrid)
    })
    
    pd.DataFrame(results_summary).to_csv(os.path.join(OUTPUT_DIR, "Data_Efficiency_Curve.csv"), index=False)

print("\n" + "="*50)
print("Experiment finished. Data saved to Data_Efficiency_Curve.csv")
print("="*50)